#02_silver_faturas.sql

Este notebook transforma bronze.faturas_pagamentos (raw/string) em silver.fact_faturas tipada, deduplicada e validada, pronta para análises de inadimplência e churn administrativo.

##Objetos

- tipar data_pagamento e valores monetários
- padroniza campos textuais (competencia/status)
- deduplica por (beneficiario_id, competencia) mantendo o mais recente por ingestion_ts
- separa rejects com motivo (valores inválidos / negativos / inconsistentes)
- persiste idempotente via MERGE usando row_hash
- roda incremental via checkpoint

##Estratégia:
- SQL-first com TEMP VIEWs (staging)
- TRY_CAST para conversões resilientes
- row_number() para dedupe determinístico
- rejects em tabela dedicada
- MERGE com row_hash

##Chave natural:
beneficiario_id, comepencia

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_rejects;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_ops;

##Tabela destino (silver + rejects)

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.fact_faturas (
  beneficiario_id STRING,
  competencia STRING,
  data_pagamento DATE,
  status_pagamento STRING,
  valor_faturado DECIMAL(18,2),
  valor_pago DECIMAL(18,2),
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.fact_faturas (
  beneficiario_id STRING,
  competencia STRING,
  data_pagamento STRING,
  status_pagamento STRING,
  valor_faturado STRING,
  valor_pago STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP
)
USING DELTA;

##Leitura incremental do bronze (checkpoint)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_faturas_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'fact_faturas'
)
SELECT *
FROM healthcare_dev.bronze.faturas_pagamentos
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS;

##Tipagem e padronização

In [0]:
CREATE OR REPLACE TEMP VIEW stg_faturas_typed AS
SELECT
  cast(beneficiario_id as string) AS beneficiario_id,
  upper(trim(cast(competencia as string))) AS competencia,
  try_cast(data_pagamento as date) AS data_pagamento,
  upper(trim(cast(status_pagamento as string))) AS status_pagamento,
  try_cast(valor_faturado as decimal(18,2)) AS valor_faturado,
  try_cast(valor_pago as decimal(18,2)) AS valor_pago,
  ingestion_ts,
  load_id,
  source_file
FROM stg_faturas_raw;

##Deduplicação deterministica (beneficiario_id, competencia)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_faturas_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY beneficiario_id, competencia
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM stg_faturas_typed
) x
WHERE rn = 1;

##Regras de qualidade (classificação + motivo)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_faturas_classified AS
SELECT
  *,
  CASE
    WHEN beneficiario_id IS NULL OR beneficiario_id = '' THEN 'MISSING_BENEFICIARIO_ID'
    WHEN competencia IS NULL OR competencia = '' THEN 'MISSING_COMPETENCIA'
    WHEN valor_faturado IS NULL THEN 'INVALID_VALOR_FATURADO'
    WHEN valor_faturado < 0 THEN 'NEGATIVE_VALOR_FATURADO'
    WHEN valor_pago IS NOT NULL AND valor_pago < 0 THEN 'NEGATIVE_VALOR_PAGO'
    WHEN valor_pago IS NOT NULL AND valor_pago > valor_faturado THEN 'VALOR_PAGO_GT_FATURADO'
    ELSE NULL
  END AS reject_reason
FROM stg_faturas_dedup;

##Persistir rejects

In [0]:
INSERT INTO healthcare_dev.silver_rejects.fact_faturas
SELECT
  beneficiario_id,
  competencia,
  cast(data_pagamento as string) AS data_pagamento,
  status_pagamento,
  cast(valor_faturado as string) AS valor_faturado,
  cast(valor_pago as string) AS valor_pago,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_faturas_classified
WHERE reject_reason IS NOT NULL;

##Valid + row_hash (para MERGE idempotente)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_faturas_valid AS
SELECT
  beneficiario_id,
  competencia,
  data_pagamento,
  status_pagamento,
  valor_faturado,
  valor_pago,
  ingestion_ts,
  load_id,
  source_file,
  sha2(concat_ws('||',
    beneficiario_id,
    competencia,
    coalesce(cast(data_pagamento as string),''),
    coalesce(status_pagamento,''),
    coalesce(cast(valor_faturado as string),''),
    coalesce(cast(valor_pago as string),'')
  ), 256) AS row_hash
FROM stg_faturas_classified
WHERE reject_reason IS NULL;

##Merge silver

In [0]:
MERGE INTO healthcare_dev.silver.fact_faturas AS tgt
USING stg_faturas_valid AS src
ON tgt.beneficiario_id = src.beneficiario_id
AND tgt.competencia = src.competencia
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Atualiza checkpoint

In [0]:
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'fact_faturas' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCCESS' AS status
FROM stg_faturas_valid;

##Checa sanidade

In [0]:
SELECT COUNT(*) AS n_silver_faturas
FROM healthcare_dev.silver.fact_faturas;

In [0]:
SELECT COUNT(*) AS n_rejects_faturas
FROM healthcare_dev.silver_rejects.fact_faturas;

##Checagem final

In [0]:
--Verifica quantidade de invalido e valor_pago_gt_faturado
SELECT reject_reason, COUNT(*) qtd
FROM healthcare_dev.silver_rejects.fact_faturas
GROUP BY reject_reason
ORDER BY qtd DESC;

In [0]:
--Verifica rejeitados
SELECT
  ROUND(100.0 * SUM(CASE WHEN reject_reason IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS pct_reject
FROM (
  SELECT * FROM healthcare_dev.silver_rejects.fact_faturas
  UNION ALL
  SELECT NULL AS beneficiario_id, NULL AS competencia, NULL AS data_pagamento, NULL AS status_pagamento,
         NULL AS valor_faturado, NULL AS valor_pago, NULL AS ingestion_ts, NULL AS load_id, NULL AS source_file,
         NULL AS reject_reason, NULL AS reject_ts
) x;